# Group G

dataset: https://www.kaggle.com/datasets/sobhanmoosavi/us-weather-events/data

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
# This might be needed in order to get plots to display in the exported HTML for submission.
# In any case, please double check that plots display properly before you submit.
import plotly.io as pio
pio.renderers.default='notebook'
df = pd.read_csv("WeatherEvents_Jan2016-Dec2022.csv")

## Picco Massimo Storico di Precipitazioni per Ogni Aeroporto (2016-2022)

Il grafico mostra il picco massimo di precipitazioni (in pollici) registrato storicamente da ciascuna stazione aeroportuale presente nel dataset. La visualizzazione evidenzia inoltre la distribuzione omogenea dei punti di rilevamento sull'intero territorio degli Stati Uniti.

In [ ]:
df_ordinato = df.sort_values(by='Precipitation(in)', ascending=False)
df_aeroporti = df_ordinato.drop_duplicates(subset=['LocationLat', 'LocationLng'])

fig = px.scatter_geo(df_aeroporti, 
                     lat="LocationLat", 
                     lon="LocationLng", 
                     scope='usa',
                     hover_name="City",
                    color_continuous_scale="Turbo",
                    title="Picco Massimo Storico di Precipitazioni per Ogni Aeroporto (2016-2022)")
fig.update_layout(hovermode=False)
fig.show()

## Distribuzione delle Precipitazioni Giornaliere in Texas - Uragano Harvey (Agosto 2017)

Il grafico mostra la distribuzione delle precipitazioni (in millimetri) registrate in Texas nel mese di agosto 2017. L'elevata concentrazione di valori estremi (outliers) evidenzia chiaramente l'impatto anomalo causato dal passaggio dell'Uragano Harvey sullo Stato.

In [ ]:
#Analisi Uragano Harvey
df['StartTime(UTC)'] = pd.to_datetime(df['StartTime(UTC)'])

df_texas = df[df['State'] == 'TX'].copy()

df_texas['Year'] = df_texas['StartTime(UTC)'].dt.year
df_texas['Month'] = df_texas['StartTime(UTC)'].dt.month
df_texas['Day'] = df_texas['StartTime(UTC)'].dt.day

df_texas = df_texas[(df_texas['Year']==2017) & (df_texas['Month'] == 8)]

df_prec = df_texas[df_texas['Precipitation(in)'] > 0]

df_prec['Precipitation(mm)'] = df_prec['Precipitation(in)'] * 25.4;

fig = px.box(df_prec,
            x='Day',
            y='Precipitation(mm)',
            title="Distribuzione delle Precipitazioni Giornaliere in Texas - Uragano Harvey (Agosto 2017)",
            labels={'Day': 'Giorno', 'Precipitation(in)': 'Precipitazioni (pollici)'},
            )
fig.update_layout(xaxis=dict(tickmode='linear', dtick=1))

fig.add_vline(
    x=27,                           # Posizione sull'asse X (27 agosto) (picco massimo)
    line_width=2,                   
    line_dash="dash",               
    line_color="red",               
    annotation_text="Picco Harvey (27 Ago)", 
    annotation_position="top left", 
    annotation_font_color="red",
    opacity=0.5
)


fig.show()


## Picco Massimo di Precipitazioni per Stato negli USA (Agosto 2017)

Il grafico mostra il picco massimo di precipitazioni per Stato nell'agosto 2017. I dati evidenziano chiaramente l'impatto eccezionale dell'Uragano Harvey sul Texas rispetto al resto del Paese.

In [ ]:
df['StartTime(UTC)'] = pd.to_datetime(df['StartTime(UTC)'])
df_agosto_2017 = df[(df['StartTime(UTC)'].dt.year == 2017) & 
                    (df['StartTime(UTC)'].dt.month == 8) & 
                    (df['Precipitation(in)'] > 0)].copy()

df_agosto_2017['Precipitation(mm)'] = df_agosto_2017['Precipitation(in)'] * 25.4

#Anomalia trovata nello stato di NY 1500mm (Impossibile) e 520mm (Impossibile), ho controllato nessun evento rilevante quel giorno

#Abbassa la soglia di pulizia per escludere l'ulteriore anomalia di 520 mm
df_agosto_2017 = df_agosto_2017[df_agosto_2017['Precipitation(mm)'] <= 450]

precipitazioni_stato = df_agosto_2017.groupby('State')['Precipitation(mm)'].max().reset_index()

fig = px.choropleth(
    precipitazioni_stato,
    locations='State',                 
    locationmode='USA-states',         
    color='Precipitation(mm)',         
    color_continuous_scale='Reds',
    range_color=[0, 430],             #Evita anomalie
    scope='usa',                       
    title="Picco Massimo di Precipitazioni per Stato negli USA (Agosto 2017)",
    labels={'Precipitation(mm)': 'Precipitazione Massima (mm)'}
)

fig.show()

## Spostamento e Intensità dell'Uragano Harvey (Agosto 2017)

Il grafico animato mappa lo spostamento e l'intensità delle precipitazioni nell'agosto 2017. La visualizzazione permette di seguire chiaramente l'andamento dell'Uragano Harvey lungo il suo percorso, evidenziando come il picco massimo del fenomeno si registri esattamente il 27 agosto.

In [ ]:
df['StartTime(UTC)'] = pd.to_datetime(df['StartTime(UTC)'])

df_agosto_2017 = df[(df['StartTime(UTC)'].dt.year == 2017) & (df['StartTime(UTC)'].dt.month == 8)]
df_agosto_2017 = df_agosto_2017[df_agosto_2017['Severity'] == 'Heavy' ]

df_agosto_2017['Day'] = df_agosto_2017['StartTime(UTC)'].dt.day

#Pulizia del dataset
df_agosto_2017['Precipitation(mm)'] = df_agosto_2017['Precipitation(in)'] * 25.4
df_agosto_2017 = df_agosto_2017[(df_agosto_2017['Precipitation(mm)'] >= 50) & (df_agosto_2017['Precipitation(mm)'] < 500) & (df_agosto_2017['Type'] == 'Rain')]
df_agosto_2017 = df_agosto_2017[df_agosto_2017['State'] != 'NY']

df_agosto_2017 = df_agosto_2017.sort_values(by='Day')


fig = px.scatter_geo(
    df_agosto_2017,
    lat="LocationLat",
    lon="LocationLng",
    color="Precipitation(mm)",        
    size="Precipitation(mm)",         
    hover_name="City",                
    animation_frame="Day",        
    scope='usa',                      
    color_continuous_scale="Reds",   
    range_color=[0, df_agosto_2017['Precipitation(mm)'].max()],
    title="Spostamento e Intensità dell'Uragano Harvey (Agosto 2017)",
    height=700,
    template="plotly_dark")


fig.show()

## Movimento di Harvey dal 26 al 29 agosto

Il grafico mostra il movimento e l'intensità dell'uragano Harvey ogni ora dla 26 al 29 agosto 2017. Si nota come l'uragno sia rimasto abbastanza fisso sul Texas, colpendo anche la Florida, senza grandi spostamenti sulla terra ferma

In [ ]:
df['StartTime(UTC)'] = pd.to_datetime(df['StartTime(UTC)'], utc=True)
df['EndTime(UTC)'] = pd.to_datetime(df['EndTime(UTC)'], utc=True)

In [ ]:
hours = pd.date_range("2017-08-26", "2017-08-29 23:00", freq="h", tz="UTC")

stations = (
    df[df["StartTime(UTC)"].dt.year == 2017]
    .dropna(subset=["LocationLat", "LocationLng"])
    .drop_duplicates("AirportCode")[["LocationLat", "LocationLng"]]
)

rain = df[
    (df["Type"] == "Rain") &
    (df["Severity"].isin(["Heavy"])) &
    (df["StartTime(UTC)"] <= hours[-1]) &
    (df["EndTime(UTC)"] >= hours[0])
].dropna(subset=["LocationLat", "LocationLng"]).copy()

rain['Precipitation(mm)'] = df['Precipitation(in)'] * 25.4

active_frames = []
for h in hours:
    active = rain[(rain["StartTime(UTC)"] <= h) & (rain["EndTime(UTC)"] > h)].copy()
    if not active.empty:
        active["CurrentHour"] = h.strftime("%Y-%m-%d %H:%M UTC")
        active_frames.append(active)

df_anim = pd.concat(active_frames, ignore_index=True)
df_anim["Severity"] = pd.Categorical(df_anim["Severity"], categories=["Heavy"], ordered=True)
df_anim = df_anim.sort_values(["CurrentHour", "Severity"])

fig = px.scatter_geo(
    df_anim,
    lat="LocationLat",
    lon="LocationLng",
    color="Precipitation(mm)",
    size="Precipitation(mm)",
    animation_frame="CurrentHour",
    hover_name="City",
    hover_data={"State": True, "Severity": False, "LocationLat": False, "LocationLng": False, "CurrentHour": False},
    color_continuous_scale="Reds",   
    range_color=[0, rain['Precipitation(mm)'].max()],
    scope="usa",
    title="Harvey — Pioggia intensa (27–29 Agosto 2017)"
)

fig.add_trace(go.Scattergeo(
    lat=stations["LocationLat"], lon=stations["LocationLng"], mode="markers",
    marker=dict(size=4, color="#37474f", opacity=0.2), hoverinfo="skip", showlegend=False
))

fig.update_traces(
    marker=dict(size=8, opacity=0.9, line=dict(width=0.5, color="white")),
    selector=dict(type="scattergeo", showlegend=True)
)

fig.update_layout(
    geo=dict(
        projection_type="albers usa", showland=True, landcolor="#1a1a2e",
        showcoastlines=True, coastlinecolor="#37474f", showsubunits=True, subunitcolor="#2e3e4e",
        bgcolor="#0f0f1a"
    ),
    paper_bgcolor="#0f0f1a", plot_bgcolor="#0f0f1a", font=dict(color="white"),
    margin=dict(l=0, r=0, t=50, b=80)
)

fig.show()

## Numero Totale di Eventi per Tipo (per Anno)

Il grafico animato mostra il numero totale di eventi meteorologici per tipologia, scorrendo anno per anno. L'utilizzo di una scala logaritmica sull'asse orizzontale permette di confrontare chiaramente fenomeni molto rari (come la grandine) con eventi estremamente frequenti (come pioggia e nebbia), garantendo la visibilità e la proporzione di tutti i dati.

Il grafico mostra che il numero di eventi meteorologici non cambia molto da un anno all'altro: 6 anni sono infatti troppo pochi per notare gli effetti del cambiamento climatico. La classifica rimane identica, nonostante qualche piccola differenza nel numero di registrazioni fra i vari anni. Nebbia e pioggia sono costantemente gli eventi più comuni ogni anno, mentre grandine e tempeste restano sempre in fondo alla lista.

I grafici precedenti mostrano l'andamento generale, ma non permettono di dare una risposta chiara a una domanda precisa: piove o nevica più di una volta nello stesso posto?

In [ ]:
df['StartTime(UTC)'] = pd.to_datetime(df['StartTime(UTC)'])
df['Year'] = df['StartTime(UTC)'].dt.year

df_filtered = df[df['Type'] != 'Precipitation']

df_counts = df_filtered.groupby(['Year', 'Type']).size().reset_index(name='Count')
df_counts = df_counts.sort_values(by=['Year', 'Count'])

max_count = df_counts['Count'].max()

fig = px.bar(
    df_counts,
    x='Count',
    y='Type',
    animation_frame='Year',       
    orientation='h',              
    title="Numero Totale di Eventi per Tipo (per Anno)",
    labels={'Count': 'Numero Totale (Scala Log)', 'Type': 'Tipo di Evento'},
    log_x=True,                   # Abilita l'asse X logaritmico
    range_x=[1, max_count * 1.5],
    color='Type',                 
)

fig.update_yaxes(categoryorder='total ascending')
fig.layout.updatemenus[0].buttons[0].args[1]['frame']['duration'] = 1200 
fig.update_layout(legend_traceorder="reversed")

fig.show()

## Percentuale Oraria degli Eventi Meteo

Il grafico illustra la distribuzione percentuale oraria degli eventi meteorologici nell'arco delle 24 ore (ora locale). Si osserva come la nebbia (Fog) presenti un picco marcato nelle prime ore del mattino, raggiungendo circa l'8.5% attorno alle 7:00, per poi decrescere rapidamente nel corso della giornata. Gli eventi di tipo Storm mostrano invece una distribuzione bimodale, con un massimo relativo in tarda mattinata e un secondo picco nel primo pomeriggio (≈6.5% alle 13:00). La grandine (Hail) segue un andamento simile agli storm, con valori crescenti nelle ore centrali della giornata. Gli eventi di Cold, Rain, Snow e Precipitation si mantengono relativamente stabili nell'intervallo 3.5–5%, senza variazioni orarie significative.

In [ ]:


def to_local(group):
    tz_name = group.name 
    group['Local_Time'] = group['StartTime(UTC)'].dt.tz_convert(tz_name).dt.tz_localize(None)
    return group

hourly_counts = df.groupby('TimeZone', group_keys=False).apply(to_local)

hourly_counts['Local_Hour'] = hourly_counts['Local_Time'].dt.hour
hourly_counts = hourly_counts.groupby(['Local_Hour', 'Type']).size().reset_index(name='Event_Count')

In [ ]:

hourly_counts['Percentage'] = hourly_counts.groupby('Type')['Event_Count'].transform(
    lambda x: (x / x.sum()) * 100
)

# Creiamo il grafico
fig = px.line(
    hourly_counts, 
    x = 'Local_Hour', 
    y = 'Percentage', 
    color = 'Type',
    title = 'Percentuale Oraria degli Eventi Meteo (Ora Locale)',
    labels = {'Local_Hour': 'Ora del Giorno (0-23)', 'Percentage': 'Percentuale [%]'},
    template = 'plotly_white'
)

fig.update_layout(xaxis=dict(tickmode='linear', tick0=0, dtick=1))

fig.show()

## Precipitazioni Medie Annuali

Il grafico mostra le precipitazioni medie annuali per stazione negli USA. Si osserva un picco nel 2017 (≈1650 mm), seguito da un calo progressivo fino al 2022, dove si registra il valore minimo del periodo (≈1100 mm). L'andamento suggerisce una tendenza decrescente che andrebbe analizzata su un periodo più lungo

In [ ]:
df["Year"] = df["StartTime(UTC)"].dt.year
df["Precipitation(mm)"] = df["Precipitation(in)"] * 25.4

mask = df["Type"] == "Rain"
annual_per_station = df.groupby(["Year", "AirportCode"])["Precipitation(mm)"].sum().reset_index()

final_yearly_data = annual_per_station.groupby("Year")["Precipitation(mm)"].mean().reset_index()


fig = px.bar(
    final_yearly_data,
    x = "Year",
    y = "Precipitation(mm)",
    title="Precipitazioni Medie Annuali per Stazione negli USA",
    labels={
        "Precipitation(mm)": "Media Pioggia Annua [mm]", 
        "Year": "Anno"
    },
    template = "plotly_white"
    )

# non mi convince questo grafico
fig.show()

## Distribuzione annuale per Neve e Nebbia

I grafici seguenti evidenziano che negli anni non c'è stato una significativo aumento di neve o nebbia.

In [ ]:
snow_year = df[df["Type"] == "Snow"].groupby("Year").size().reset_index(name = "SnowCount")
fog_year = df[df["Type"] == "Fog"].groupby("Year").size().reset_index(name = "FogCount")

fig = make_subplots(
    rows=2, cols=1, 
    shared_xaxes=True, 
    vertical_spacing=0.1,
    subplot_titles=("Distribuzione Annuale Neve", "Distribuzione Annuale Nebbia")
)


fig.add_trace(
    go.Bar(
        x = snow_year["Year"],
        y = snow_year["SnowCount"], 
        name = "Neve",
        marker_color = "skyblue"
    ),
    row=1, col=1
)


fig.add_trace(
    go.Bar(
        x = fog_year["Year"],
        y = fog_year["FogCount"], 
        name = "Nebbia",
        marker_color = "lightgrey"
    ),
    row=2, col=1
)

fig.update_layout(
    title_text = "Analisi Eventi Meteo per Anno",
    template = "plotly_white",
    showlegend = False,
    height = 700
)

fig.update_yaxes(title_text="Conteggio Neve", row=1, col=1)
fig.update_yaxes(title_text="Conteggio Nebbia", row=2, col=1)
fig.update_xaxes(title_text="Anno", row=2, col=1)

fig.show()

## Heatmap della severità per evento

Gli eventi con gravità "Severe" sono fog, storm e cold.

Per la nebbia si nota una maggiore frequenza di eventi estremi nel periodo invernale, con un aumento generalizzato dal 2019

In [ ]:
mask = (df['Severity'] == 'Severe') & (df["Type"] == "Fog")
df_severe_fog = df[mask].copy()
#il copy va usato perche pandas ritorna una vista del dataseto originale se dopo modifichiamo df_severe potrebbe dare errore

df_severe_fog['Month'] = df_severe_fog['StartTime(UTC)'].dt.month
#trasformandolo in datetime di pandas ci possiamo accedere con dt e abbiamo le sue funzionalità

#Creiamo la heatmap

#Contiamo per ongi combinazione di mese-anno il numero di eventi severe (collassa tutte le righe Gennaio 2016 in una sola contando il numero di occorrenze)
heatmap = df_severe_fog.groupby(['Year','Month']).size().reset_index(name='Count')
#Quindi dopo qui abbiamo le righe singole che sono Gennaio 2016 / febbraio 2016 ... agosto 2016 e per ognuna di queste righe uniche abbiamo il numero di count 

#Creiamo una tabella pivot da passare alla heatmap
pivot_table = heatmap.pivot(index='Year', columns='Month', values='Count').fillna(0)

fig = px.imshow(pivot_table, 
                title="Stagionalità e Frequenza degli Eventi Estremi di Nebbia (2016-2022)",
                labels=dict(x="Mese", y="Anno", color="N eventi Estremi"),
                x=['Gen', 'Feb', 'Mar', 'Apr', 'Mag', 'Giu', 'Lug', 'Ago', 'Set', 'Ott', 'Nov', 'Dic']
                );

fig.show()

Per il freddo estremo si nota una tendenza a decrescente nel tempo, con però picchi più intensi. Da notare che sono prsenti eventi di freddo estremo in piena estate, potrebbero essere errori del dataset, ma non c'è una colonna per la temperatura o il soleggiamento per verificarlo.

In [ ]:
mask = (df['Severity'] == 'Severe') & (df["Type"] == "Cold")
df_severe_cold = df[mask].copy()
#il copy va usato perche pandas ritorna una vista del dataseto originale se dopo modifichiamo df_severe potrebbe dare errore

df_severe_cold['Month'] = df_severe_cold['StartTime(UTC)'].dt.month
#trasformandolo in datetime di pandas ci possiamo accedere con dt e abbiamo le sue funzionalità

#Creiamo la heatmap

#Contiamo per ongi combinazione di mese-anno il numero di eventi severe (collassa tutte le righe Gennaio 2016 in una sola contando il numero di occorrenze)
heatmap = df_severe_cold.groupby(['Year','Month']).size().reset_index(name='Count')
#Quindi dopo qui abbiamo le righe singole che sono Gennaio 2016 / febbraio 2016 ... agosto 2016 e per ognuna di queste righe uniche abbiamo il numero di count 

#Creiamo una tabella pivot da passare alla heatmap
pivot_table = heatmap.pivot(index='Year', columns='Month', values='Count').fillna(0)

fig = px.imshow(pivot_table, 
                title="Stagionalità e Frequenza degli Eventi Estremi di Fereddo (< -23.7°C) (2016-2022)",
                labels=dict(x="Mese", y="Anno", color="N eventi Estremi"),
                x=['Gen', 'Feb', 'Mar', 'Apr', 'Mag', 'Giu', 'Lug', 'Ago', 'Set', 'Ott', 'Nov', 'Dic']
                );

fig.show()

Per le tempeste si nota una minore frequenza nei mesi estivi, con una leggera tendenza all'aumento nel tempo

In [ ]:
mask = (df['Severity'] == 'Severe') & (df["Type"] == "Storm")
df_severe_storm = df[mask].copy()
#il copy va usato perche pandas ritorna una vista del dataseto originale se dopo modifichiamo df_severe potrebbe dare errore

df_severe_storm['Month'] = df_severe_storm['StartTime(UTC)'].dt.month
#trasformandolo in datetime di pandas ci possiamo accedere con dt e abbiamo le sue funzionalità

#Creiamo la heatmap

#Contiamo per ongi combinazione di mese-anno il numero di eventi severe (collassa tutte le righe Gennaio 2016 in una sola contando il numero di occorrenze)
heatmap = df_severe_storm.groupby(['Year','Month']).size().reset_index(name='Count')
#Quindi dopo qui abbiamo le righe singole che sono Gennaio 2016 / febbraio 2016 ... agosto 2016 e per ognuna di queste righe uniche abbiamo il numero di count 

#Creiamo una tabella pivot da passare alla heatmap
pivot_table = heatmap.pivot(index='Year', columns='Month', values='Count').fillna(0)

fig = px.imshow(pivot_table, 
                title="Stagionalità e Frequenza degli Eventi Estremi di Tempeste (2016-2022)",
                labels=dict(x="Mese", y="Anno", color="N eventi Estremi"),
                x=['Gen', 'Feb', 'Mar', 'Apr', 'Mag', 'Giu', 'Lug', 'Ago', 'Set', 'Ott', 'Nov', 'Dic']
                );

fig.show()

## Percentuale di intensità per tipo di evento

Ci aspettiamo un trend di aumento degli eventi intensi, ma le aspettative vengono confermate solo per la nebbia, per gli altri eventi non emerge un trend significativo

In [ ]:
for e in df.Type.unique():
    df_type = df[df['Type'] == e].copy()
    
    # Conta eventi per Severity e Year
    df_count = df_type.groupby(['Year', 'Severity']).size().reset_index(name='Count')
    
    # Calcola percentuale per anno
    df_count['Pct'] = df_count.groupby('Year')['Count'].transform(lambda x: x / x.sum() * 100)
    
    severity_order = ['Light', 'Moderate', 'Heavy', 'Severe']
    
    fig = px.bar(
        df_count,
        x='Year',
        y='Pct',
        color='Severity',
        barmode='relative',  # stacked 100%
        title=f'Distribuzione Intensità - {e}',
        labels={'Pct': 'Eventi [%]', 'Year': 'Anno', 'Severity': 'Intensità'},
        category_orders={'Severity': severity_order}
    )
    fig.update_layout(
        xaxis=dict(tickmode='linear', dtick=1),
        yaxis=dict(ticksuffix='%'),
        legend=dict(traceorder='reversed')
    )
    fig.show()

## Precipitazione media in funzione della durata media nel tempo 

In [ ]:
noaa_regions = {
    # Northeast
    'CT':'Northeast','DE':'Northeast','ME':'Northeast','MD':'Northeast',
    'MA':'Northeast','NH':'Northeast','NJ':'Northeast','NY':'Northeast',
    'PA':'Northeast','RI':'Northeast','VT':'Northeast',
    # East North Central
    'IA':'East North Central','MI':'East North Central',
    'MN':'East North Central','WI':'East North Central',
    # Central
    'IL':'Central','IN':'Central','KY':'Central','MO':'Central',
    'OH':'Central','TN':'Central','WV':'Central',
    # Southeast
    'AL':'Southeast','FL':'Southeast','GA':'Southeast',
    'NC':'Southeast','SC':'Southeast','VA':'Southeast',
    # West North Central
    'MT':'West North Central','NE':'West North Central',
    'ND':'West North Central','SD':'West North Central','WY':'West North Central',
    # South
    'AR':'South','KS':'South','LA':'South','MS':'South','OK':'South','TX':'South',
    # Southwest
    'AZ':'Southwest','CO':'Southwest','NM':'Southwest','UT':'Southwest',
    # Northwest
    'ID':'Northwest','OR':'Northwest','WA':'Northwest',
    # West
    'CA':'West','NV':'West',
}
# fonte: https://psl.noaa.gov/data/usclimdivs/data/

df['Region'] = df['State'].map(noaa_regions).fillna('Other')

Per la pioggia, gli stati del Nord-Ovset tendono ad avere pioggielunge ma poco precipitose, mentre gli stati del Sud e Sud –Est hanno pioggie brevi ma intense. Conforntando direttamente il 2016 col 2022 si nota una diminuziuoine delle precipitazioni e della durata per le regioni tranne il Northwest

In [ ]:
df_rain = df[df['Type'] == 'Rain'].copy()
df_rain['DurationHrs'] = (df_rain['EndTime(UTC)'] - df_rain['StartTime(UTC)']).dt.total_seconds() / 3600
df_rain = df_rain[(df_rain['DurationHrs'] > 0) & (df_rain['Precipitation(mm)'] > 0)]

cap_d = df_rain['DurationHrs'].quantile(0.99)
cap_p = df_rain['Precipitation(mm)'].quantile(0.99)
df_rain = df_rain[(df_rain['DurationHrs'] <= cap_d) & (df_rain['Precipitation(mm)'] <= cap_p)]

agg = df_rain.groupby(['State', 'Year', 'Region']).agg(
    AvgDuration=('DurationHrs', 'mean'),
    AvgPrecip=('Precipitation(mm)', 'mean'),
    Count=('EventId', 'count')
).reset_index()

fig = px.scatter(
    agg,
    x='AvgDuration',
    y='AvgPrecip',
    size='Count',
    color='Region',
    animation_frame='Year',
    hover_name='State',
    title='Eventi di pioggia per Stato: Durata vs Precipitazioni nel tempo',
    labels={
        'AvgDuration': 'Durata media [h]',
        'AvgPrecip': 'Precipitazione media [mm]',
        'Count': 'Numero Eventi'
    },
    size_max=50,
    template='plotly_dark'
)

fig.update_layout(
    xaxis=dict(range=[0, agg['AvgDuration'].quantile(0.98)]),
    yaxis=dict(range=[0, agg['AvgPrecip'].quantile(0.98)])
)
fig.show()

Per la neve, gli stati del East North Central hanno precipitazioni nevose più lunghe, mentre al South, Southeast, West si
hanno precipitazioni brevi.​ Tendono ad esserci tante oscillazionio tra un  anno e l'altro per gli stati con pochi eventi

In [ ]:
df_snow = df[df['Type'] == 'Snow'].copy()
df_snow['DurationHrs'] = (df_snow['EndTime(UTC)'] - df_snow['StartTime(UTC)']).dt.total_seconds() / 3600
df_snow = df_snow[(df_snow['DurationHrs'] > 0) & (df_snow['Precipitation(mm)'] > 0)]

cap_d = df_snow['DurationHrs'].quantile(0.99)
cap_p = df_snow['Precipitation(mm)'].quantile(0.99)
df_snow = df_snow[(df_snow['DurationHrs'] <= cap_d) & (df_snow['Precipitation(mm)'] <= cap_p)]

agg = df_snow.groupby(['State', 'Year', 'Region']).agg(
    AvgDuration=('DurationHrs', 'mean'),
    AvgPrecip=('Precipitation(mm)', 'mean'),
    Count=('EventId', 'count')
).reset_index()

fig = px.scatter(
    agg,
    x='AvgDuration',
    y='AvgPrecip',
    size='Count',
    color='Region',
    animation_frame='Year',
    hover_name='State',
    title='Eventi di neve per Stato: Durata vs Precipitazioni nel tempo',
    labels={
        'AvgDuration': 'Durata media [h]',
        'AvgPrecip': 'Precipitazione media [mm]',
        'Count': 'Numero Eventi'
    },
    size_max=50,
    template='plotly_dark'
)

fig.update_layout(
    xaxis=dict(range=[0, agg['AvgDuration'].quantile(0.98)]),
    yaxis=dict(range=[0, agg['AvgPrecip'].quantile(0.98)])
)
fig.show()